# 30_feature_selection_main
**NOVA IMS – DSAA 2025/26**  
**Cars 4 You – Feature Selection (kompatibel mit 00/10/20 Notebooks)**  
*Generated on:* 2025-10-17

Dieses Notebook erkennt automatisch eure **Pfade/Variablen** aus `00_`, `10_`, `20_` 
und nutzt die gleichen Konventionen (DATA_DIR, TRAIN/TEST, TARGET). 
Es implementiert **Filter/Wrapper/Embedded**-Selektion mit konsistenter 10-fold CV.

## Setup & Auto-Erkennung

In [9]:
# ===== Projektpfade (auto-detected; bei Bedarf anpassen) =====
from pathlib import Path

DATA_DIR  = Path("../data")
TRAIN_FILE = DATA_DIR / "processed_train_data.csv"
TEST_FILE  = DATA_DIR / "test.csv"
TARGET = "price"

# Optional: ID-Spalten (falls in früheren Notebooks definiert)
ID_COLS = []
# TODO: Falls in 20er Notebook vorhanden, hier ergänzen, z.B.: ID_COLS = ["carID"]

print("DATA_DIR :", DATA_DIR)
print("TRAIN   :", TRAIN_FILE)
print("TEST    :", TEST_FILE)
print("TARGET  :", TARGET)

DATA_DIR : ../data
TRAIN   : ../data/processed_train_data.csv
TEST    : ../data/test.csv
TARGET  : price


In [ ]:
# ================
# Imports
# ================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import json

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LassoCV, ElasticNetCV, RidgeCV
from sklearn.feature_selection import RFE, SelectKBest, f_regression, mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from scipy.stats import spearmanr

## Daten & Preprocessing

In [11]:
# =======================
# Daten laden
# =======================
train = pd.read_csv(TRAIN_FILE)
test  = pd.read_csv(TEST_FILE)

print("Train shape:", train.shape, " | Test shape:", test.shape)
assert TARGET in train.columns, f"Zielvariable '{TARGET}' nicht gefunden!"

# Train-Features vs Target
X = train.drop(columns=[TARGET] + ID_COLS if len(ID_COLS)>0 else [TARGET])
y = train[TARGET].copy()

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print("num:", len(num_cols), " | cat:", len(cat_cols))

Train shape: (75973, 14)  | Test shape: (32567, 13)
num: 9  | cat: 4


In [12]:
# =============================================
# Preprocessing (aligned with 20er Notebook Konventionen)
# =============================================
numeric_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler(with_mean=True, with_std=True)),
])

categorical_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

In [13]:
# ==================================
# CV & Scorer
# ==================================
RANDOM_STATE = 42
CV_FOLDS = 10
N_JOBS = -1

kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scorer_mae  = make_scorer(mean_absolute_error, greater_is_better=False)
scorer_rmse = make_scorer(lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)), greater_is_better=False)
scorer_r2   = make_scorer(r2_score, greater_is_better=True)

def cv_report(model, X, y, name="model"):
    scores_mae  = cross_val_score(model, X, y, scoring=scorer_mae, cv=kf, n_jobs=N_JOBS)
    scores_rmse = cross_val_score(model, X, y, scoring=scorer_rmse, cv=kf, n_jobs=N_JOBS)
    scores_r2   = cross_val_score(model, X, y, scoring=scorer_r2, cv=kf, n_jobs=N_JOBS)
    print(f"[{name}]  MAE (↓): { -scores_mae.mean():.3f} ± {scores_mae.std():.3f} | "
          f"RMSE (↓): { -scores_rmse.mean():.3f} ± {scores_rmse.std():.3f} | "
          f"R² (↑): { scores_r2.mean():.3f} ± {scores_r2.std():.3f}")

In [14]:
# =====================
# Baseline: LinearReg
# =====================
pipe_lin = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LinearRegression())
])

cv_report(pipe_lin, X, y, name="Baseline LinearRegression")

[Baseline LinearRegression]  MAE (↓): 2517.617 ± 24.362 | RMSE (↓): 4081.683 ± 188.328 | R² (↑): 0.824 ± 0.011


## Filter-Methoden

In [15]:
# ===========================================
# Filter Scores: f_regression, MI, Spearman
# ===========================================
_ = preprocessor.fit(X)

def get_feature_names(preprocessor, num_cols, cat_cols):
    feature_names = []
    feature_names += [f"NUM::{c}" for c in num_cols]
    ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
    ohe_names = list(ohe.get_feature_names_out(cat_cols))
    feature_names += [f"CAT::{n}" for n in ohe_names]
    return feature_names

feature_names = get_feature_names(preprocessor, num_cols, cat_cols)
X_trans = preprocessor.transform(X)

# f_regression
f_selector = SelectKBest(score_func=f_regression, k="all").fit(X_trans, y)
scores_f = f_selector.scores_

# mutual information
mi = mutual_info_regression(X_trans, y, random_state=RANDOM_STATE)
scores_mi = mi

# Spearman (auf transformiertem Raum)
def spearman_scores(X_arr, y_arr):
    s = []
    for j in range(X_arr.shape[1]):
        coef, _ = spearmanr(X_arr[:, j], y_arr)
        s.append(0.0 if np.isnan(coef) else abs(coef))
    return np.array(s)

scores_spear = spearman_scores(X_trans, y)

filter_df = pd.DataFrame({
    "feature": feature_names,
    "score_f": scores_f,
    "score_mi": scores_mi,
    "score_spear": scores_spear
}).fillna(0.0)

for col in ["score_f", "score_mi", "score_spear"]:
    filter_df[f"rank_{col}"] = filter_df[col].rank(ascending=False, method="average")

filter_df["rank_mean"] = filter_df[[
    "rank_score_f", "rank_score_mi", "rank_score_spear"
]].mean(axis=1)

filter_df = filter_df.sort_values("rank_mean")
filter_df.head(15)

,feature,score_f,score_mi,score_spear,rank_score_f,rank_score_mi,rank_score_spear,rank_mean
5,NUM::engineSize,43924.555814,0.397524,0.558592,1.0,1.0,3.0,1.666667
1,NUM::year,21750.266919,0.337625,0.588676,3.0,4.0,1.0,2.666667
225,CAT::transmission_manual,28200.393125,0.203761,0.568534,2.0,6.0,2.0,3.333333
2,NUM::mileage,14581.336025,0.344343,0.508825,5.0,3.0,4.0,4.000000
4,NUM::mpg,6184.273205,0.380588,0.370952,8.0,2.0,6.0,5.333333
229,CAT::transmission_semi-auto,14648.288449,0.124297,0.432083,4.0,8.0,5.0,5.666667
12,CAT::Brand_Mercedes-Benz,9423.533239,0.113329,0.357209,6.0,9.0,7.0,7.333333
3,NUM::tax,6727.701515,0.181136,0.297741,7.0,7.0,9.0,7.666667
13,CAT::Brand_Opel,5093.515124,0.070386,0.308434,9.0,10.0,8.0,9.000000
0,NUM::carID,3079.761362,0.236260,0.206011,13.0,5.0,16.0,11.333333


In [16]:
# ============================
# Top-K per CV finden (Filter)
# ============================
def eval_topk_by_filter(k, ranking_df):
    keep = set(ranking_df.head(k)["feature"])
    def _masker_from_keep(X):
        Xt = preprocessor.transform(X)
        idx = [i for i, name in enumerate(feature_names) if name in keep]
        return Xt[:, idx]
    mask = FunctionTransformer(_masker_from_keep, feature_names_out="one-to-one")
    pipe = Pipeline([("mask", mask), ("model", LinearRegression())])
    scores = cross_val_score(pipe, X, y, scoring=scorer_mae, cv=kf, n_jobs=N_JOBS)
    return -scores.mean()

candidate_ks = list(range(5, min(60, len(feature_names))+1, 5))
results = []
for k in candidate_ks:
    mae = eval_topk_by_filter(k, filter_df)
    results.append((k, mae))

res_df = pd.DataFrame(results, columns=["k", "cv_mae"]).sort_values("cv_mae")
best_k = int(res_df.iloc[0]["k"])
best_filter_features = set(filter_df.head(best_k)["feature"])
print("Best k (Filter):", best_k, "| Selected:", len(best_filter_features))
res_df.head(10)

Best k (Filter): 60 | Selected: 60


,k,cv_mae
11,60,2837.946706
10,55,2865.531117
9,50,2883.936115
8,45,2953.971154
7,40,2991.737493
6,35,3021.086454
5,30,3111.376709
4,25,3208.229831
3,20,3266.113137
2,15,3335.930264


## Wrapper-Methoden (RFE)

In [19]:
# =====================================================
# RFE ULTRA-FAST  (< ~2 Min bei üblichen Datengrößen)
# =====================================================
# Voraussetzungen:
# - preprocessor, X, y
# - kf (oder definiere hier schnell kf=KFold(n_splits=5, shuffle=True, random_state=42))
# - feature_names, und idealerweise filter_df (mit 'rank_mean')

from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import numpy as np, pandas as pd

# ------------------ SPEED-KNOBS ------------------
CV_FOLDS_RFE = 5         # 5 statt 10 Folds für die RFE-Auswahl
M_TOP = 60               # nur Top-60 Spalten in die RFE geben
K_GRID = None            # oder z.B. [15, 30, 45, 60] – sehr wenige k-Werte
STEP = 0.5               # aggressives Entfernen (schneller)
DO_RF_QUICK_CHECK = True # optionaler schneller RF-Check am Ende
# --------------------------------------------------

if 'kf' not in globals():
    kf = KFold(n_splits=CV_FOLDS_RFE, shuffle=True, random_state=42)

# 0) Einmal transformieren
_ = preprocessor.fit(X)
X_t_full = preprocessor.transform(X)

# 1) Pre-Pruning (Top-M)
if 'filter_df' in globals():
    pre_keep = set(filter_df.sort_values("rank_mean").head(M_TOP)["feature"])
else:
    skb = SelectKBest(score_func=f_regression, k="all").fit(X_t_full, y)
    tmp = pd.DataFrame({"feature": feature_names, "score": skb.scores_}).fillna(0.0)
    pre_keep = set(tmp.sort_values("score", ascending=False).head(M_TOP)["feature"])

pre_idx   = np.array([i for i, n in enumerate(feature_names) if n in pre_keep])
X_t       = X_t_full[:, pre_idx]
names_rfe = [feature_names[i] for i in pre_idx]

# 2) CV-Splits vorab
kf_splits = list(kf.split(X_t, y))

# 3) Schnelle RFE nur mit LinearRegression (sehr schnell + stabil)
def rfe_ultra(base_estimator, X_t, y, names, kf_splits,
              step=0.5, min_features_to_select=5, k_grid=None):
    n_features = X_t.shape[1]
    if k_grid is None:
        # Minimales Raster (4-5 Punkte)
        k_vals = np.unique(np.clip(
            (np.geomspace(min_features_to_select, n_features, num=5)).astype(int),
            min_features_to_select, n_features
        ))
    else:
        k_vals = np.unique(np.array(k_grid, dtype=int))

    best = (None, np.inf, None)
    recs = []

    for n_keep in k_vals:
        rfe = RFE(estimator=clone(base_estimator),
                  n_features_to_select=int(n_keep), step=step)
        rfe.fit(X_t, y)
        support = rfe.support_

        # schnelle CV direkt auf Arrays
        maes = []
        for tr, te in kf_splits:
            est = clone(base_estimator)
            est.fit(X_t[tr][:, support], y.iloc[tr] if hasattr(y, "iloc") else y[tr])
            pred = est.predict(X_t[te][:, support])
            maes.append(mean_absolute_error(y.iloc[te] if hasattr(y, "iloc") else y[te], pred))
        mae = float(np.mean(maes))
        recs.append((int(n_keep), mae, support))
        if mae < best[1]:
            best = (int(n_keep), mae, support)

    df = pd.DataFrame([(k, m) for k, m, _ in recs], columns=["n_keep", "cv_mae"]).sort_values("cv_mae")
    print(f"[RFE-ULTRA] Best n_keep={best[0]} | MAE={best[1]:.3f} | tested k={list(df['n_keep'])}")
    return best[0], best[1], best[2], df

print("RFE (LinearRegression, ULTRA)")
bestk_lin, mae_lin, sup_lin, df_lin = rfe_ultra(
    LinearRegression(), X_t, y, names_rfe, kf_splits,
    step=STEP, min_features_to_select=max(5, int(M_TOP*0.15)), k_grid=K_GRID
)

# 4) Finales Support-Set aus LR-RFE
rfe_selected_features = {names_rfe[i] for i, v in enumerate(sup_lin) if v}
print("RFE selected features:", len(rfe_selected_features))

# 5) Optional: sehr schneller RF-Check (gleiche Auswahl, nur 3-fold, um <2 Min zu bleiben)
if DO_RF_QUICK_CHECK:
    kf3 = KFold(n_splits=3, shuffle=True, random_state=42)
    sup_idx = np.array([i for i, v in enumerate(sup_lin) if v])  # Indizes relativ zu names_rfe / X_t
    rf_quick = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

    maes = []
    for tr, te in kf3.split(X_t, y):
        rf_quick.fit(X_t[tr][:, sup_idx], y.iloc[tr] if hasattr(y,"iloc") else y[tr])
        pred = rf_quick.predict(X_t[te][:, sup_idx])
        maes.append(mean_absolute_error(y.iloc[te] if hasattr(y,"iloc") else y[te], pred))
    print(f"[RF quick-check] 3-fold MAE: {np.mean(maes):.3f}")

# Ergebnis: rfe_selected_features ist das reduzierte Set (Transform-Namen)
list(sorted(rfe_selected_features))[:10]


RFE (LinearRegression, ULTRA)
[RFE-ULTRA] Best n_keep=60 | MAE=2837.947 | tested k=[60, 37, 23, 14, 9]
RFE selected features: 60
[RF quick-check] 3-fold MAE: 1498.136


['CAT::Brand_BMW',
 'CAT::Brand_Ford',
 'CAT::Brand_Hyundai',
 'CAT::Brand_Mercedes-Benz',
 'CAT::Brand_Opel',
 'CAT::Brand_Toyota',
 'CAT::Brand_Škoda',
 'CAT::fuelType_hybrid',
 'CAT::fuelType_petrol',
 'CAT::model_3 Series']

## Embedded-Methoden

In [21]:
# =====================================
# Embedded: LassoCV / ElasticNetCV / RidgeCV  (FIX + FAST)
# =====================================
from sklearn.linear_model import LassoCV, ElasticNetCV, RidgeCV
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
import numpy as np, pandas as pd

# --- Einheitlicher Fit & Transform ---
_ = preprocessor.fit(X)
X_t = preprocessor.transform(X)

# --- Lasso / ElasticNet im transformierten Raum ---
# Tipp: bei Bedarf speed-up: cv=5, n_alphas=50 (Default ~100)
lasso = LassoCV(cv=kf, random_state=42, n_jobs=-1).fit(X_t, y)
enet  = ElasticNetCV(cv=kf, random_state=42, n_jobs=-1).fit(X_t, y)

def nonzero_features(model, names, atol=1e-8):
    coef = getattr(model, "coef_", None)
    if coef is None:
        return set(names)
    nz = np.where(np.abs(coef) > atol)[0]
    return {names[i] for i in nz}

lasso_feats = nonzero_features(lasso, feature_names)
enet_feats  = nonzero_features(enet,  feature_names)

# --- RidgeCV + Permutation Importance IM TRANSFORMIERTEN RAUM ---
# WICHTIG: Trainiere RidgeCV direkt auf X_t
ridge = RidgeCV(cv=kf).fit(X_t, y)

# Permutation Importance auf X_t (passt zu feature_names)
# Für Speed ggf. n_repeats=5 statt 10
perm = permutation_importance(ridge, X_t, y, n_repeats=10, random_state=42, n_jobs=-1)

ridge_imp = pd.DataFrame({
    "feature": feature_names,                       # transformierte Namen (NUM::, CAT::…)
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

# Schwelle: oberes Quartil
q25 = ridge_imp["importance"].quantile(0.25)
ridge_feats = set(ridge_imp.loc[ridge_imp["importance"] > q25, "feature"])

print(
    f"Lasso non-zero: {len(lasso_feats)} | "
    f"ElasticNet non-zero: {len(enet_feats)} | "
    f"Ridge PI>Q25: {len(ridge_feats)}"
)


Lasso non-zero: 93 | ElasticNet non-zero: 126 | Ridge PI>Q25: 177


## Fusion & Finale Bewertung

In [22]:
# =============================
# Fusion der Feature-Sets
# =============================
embedded_union = lasso_feats | enet_feats | ridge_feats
core = (best_filter_features & rfe_selected_features & embedded_union)
FINAL_MIN = 12
final = set(core)

if len(final) < FINAL_MIN:
    for feat in filter_df["feature"]:
        if feat not in final:
            final.add(feat)
            if len(final) >= FINAL_MIN:
                break

print("Core:", len(core), " | Final:", len(final))
sorted(list(final))[:20]

Core: 60  | Final: 60


['CAT::Brand_BMW',
 'CAT::Brand_Ford',
 'CAT::Brand_Hyundai',
 'CAT::Brand_Mercedes-Benz',
 'CAT::Brand_Opel',
 'CAT::Brand_Toyota',
 'CAT::Brand_Škoda',
 'CAT::fuelType_hybrid',
 'CAT::fuelType_petrol',
 'CAT::model_3 Series',
 'CAT::model_4 Series',
 'CAT::model_5 Series',
 'CAT::model_A-Class',
 'CAT::model_A5',
 'CAT::model_Adam',
 'CAT::model_Astra',
 'CAT::model_Aygo',
 'CAT::model_C-Class',
 'CAT::model_Citigo',
 'CAT::model_Corsa']

In [24]:
# =============================================
# FAST: Finale CV-Performance ohne Pipeline-Overhead
# =============================================
import time
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import numpy as np

t0 = time.time()

# 1) Einmalig fitten/transformieren + Final-Indices bestimmen
_ = preprocessor.fit(X)
X_t = preprocessor.transform(X)                     # transformierter Raum
# feature_names muss hier schon existieren (aus deiner Filter-Zelle), sonst neu bauen:
#   ohe = preprocessor.named_transformers_["cat"].named_steps["ohe"]
#   feature_names = [f"NUM::{c}" for c in num_cols] + [f"CAT::{n}" for n in ohe.get_feature_names_out(cat_cols)]
idx_final = np.array([i for i, n in enumerate(feature_names) if n in final])
X_final = X_t[:, idx_final]                         # nur finale Features

# 2) Schnelle CV (5-fold) direkt auf Arrays
kf_fast = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_report_fast(estimator, Xf, y, name):
    maes, rmses, r2s = [], [], []
    for tr, te in kf_fast.split(Xf, y):
        est = estimator.__class__(**estimator.get_params())
        est.fit(Xf[tr], y.iloc[tr] if hasattr(y, "iloc") else y[tr])
        pred = est.predict(Xf[te])
        yt = y.iloc[te] if hasattr(y, "iloc") else y[te]
        maes.append(mean_absolute_error(yt, pred))
        rmses.append(np.sqrt(mean_squared_error(yt, pred)))
        r2s.append(r2_score(yt, pred))
    print(f"[{name}]  MAE (↓): {np.mean(maes):.3f} ± {np.std(maes):.3f} | "
          f"RMSE (↓): {np.mean(rmses):.3f} ± {np.std(rmses):.3f} | "
          f"R² (↑): {np.mean(r2s):.3f} ± {np.std(r2s):.3f}")

print("\n=== Final Feature Set – FAST CV (5-fold, pretransformed) ===")
cv_report_fast(LinearRegression(), X_final, y, name="Final LinearRegression")

# 3) Schneller RF-Lauf (200 Trees) – für Finale gern wieder 600 verwenden
rf_fast = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
cv_report_fast(rf_fast, X_final, y, name="Final RandomForest (200)")

print(f"Done in {time.time() - t0:.1f}s")



=== Final Feature Set – FAST CV (5-fold, pretransformed) ===
[Final LinearRegression]  MAE (↓): 2838.585 ± 32.393 | RMSE (↓): 4730.620 ± 206.353 | R² (↑): 0.764 ± 0.011
[Final RandomForest (200)]  MAE (↓): 1464.336 ± 20.784 | RMSE (↓): 2588.943 ± 249.098 | R² (↑): 0.929 ± 0.012
Done in 71.1s


## Export

In [ ]:
# =====================
# Export der Auswahl
# =====================
ARTIFACTS = Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True, parents=True)

meta = {
    "generated_on": "2025-10-17",
    "cv_folds": 10,
    "primary_metric": "MAE",
    "final_feature_count": len(keep_final),
    "final_features": sorted(list(keep_final)),
    "notes": "Feature-Namen im transformierten Raum (NUM::col, CAT::<col>_<level>)."
}

out_file = ARTIFACTS / "selected_features.json"
out_file.write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print("Gespeichert:", out_file.resolve())

Gespeichert: /Users/karaca/src/MachineLearningProject-NOVAIMS2025/notebooks/artifacts/selected_features.json
